# Cell 1: gold split

In [ ]:
import os
import glob, json, time
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

train = pd.read_csv(f"{ROOT}/train.csv")
gold = train[train[LABELS].notna().all(axis=1)].reset_index(drop=True)

rng = np.random.default_rng(42)
dev_ids = set(rng.choice(gold["StudyInstanceUID"], size=20, replace=False))
gold["split"] = np.where(gold["StudyInstanceUID"].isin(dev_ids), "dev", "holdout")
print(gold.groupby("split")[LABELS].sum().astype(int).T.to_string())
gold[["StudyInstanceUID", "split"]].to_csv("/kaggle/working/gold_split.csv", index=False)

## Decisions made from dev errors
- D1: severity 0 with empty evidence is treated as unmentioned (score 0.5). 8B dev macro 0.814 -> 0.820.
- D2: prompt says severity 0 needs an explicit quote, and a finding that is present with no
  severity words is severity 3.
- D3: a severity 0 answer whose evidence is not found verbatim in the report is treated as
  unmentioned (the model wrote "nicht erwaehnt" = "not mentioned" as evidence). 0.811 -> 0.814.

## Engineering notes
- 8B split across two T4s: about 10 h for all reports, out-of-memory at batch 20, then an
  illegal memory access. Switched to one model copy per GPU.
- Qwen3-4B fp16, one copy per GPU: dev macro 0.752, about 3 h. Too weak.
- Qwen3-8B NF4 4-bit, one copy per GPU: dev macro 0.811 (0.814 with D3), about 9 h.
  Chosen. Full run done in 03b, split over two accounts.

## Known issues in silver labels
- Size words are sometimes misgraded in non-English reports (Bulgarian "malka" = small
  Baker's cyst got severity 4).
- Evidence quotes often exceed the 8-word cap, especially in Cyrillic and Greek. Parse
  failures in the throughput test were all outputs cut off at 400 tokens.

# Cell 2: two copies of 4B

In [ ]:
import torch, transformers
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM

cfgs = [p for top in os.listdir("/kaggle/input") if top != "competitions"
        for p in glob.glob(f"/kaggle/input/{top}/**/config.json", recursive=True)]
MODEL_DIR = [os.path.dirname(c) for c in cfgs if "/4b/" in c][0]
print(MODEL_DIR, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count())

toks, models = [], []
for i in range(torch.cuda.device_count()):
    t = AutoTokenizer.from_pretrained(MODEL_DIR)
    t.padding_side = "left"
    toks.append(t)
    models.append(AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.float16).to(f"cuda:{i}").eval())
print({i: f"{torch.cuda.memory_allocated(i) / 1e9:.1f} GB" for i in range(torch.cuda.device_count())})

# Cell 3: prompt, parsing, scoring, parallel runner

In [ ]:
SYSTEM = """You read knee MRI radiology reports and extract 12 findings for one knee.
Reports can be in any language. Answer only with JSON, no other text.

For each finding return [severity, certainty, evidence].
severity:
  0 = the report explicitly says it is normal, intact or absent; quote those words
  1 = minimal, trace, degenerative, intrasubstance signal, chronic or old
  2 = mild, small, low grade, grade 1
  3 = moderate, partial tear, grade 2, or a tear or finding that is present with no severity words
  4 = severe, large, massive, complete or full-thickness, grade 3 or 4, displaced
  null = the report does not mention this finding; never use 0 for an unmentioned finding
certainty: 2 = definite, 1 = probable or likely, 0 = possible, suspected, cannot be excluded
evidence: at most 8 words copied from the report in its original language, or ""

Findings:
ACL, MCL: injury or tear of that ligament.
Medial Meniscus, Lateral Meniscus: tear or abnormality of that meniscus. Signal that does not
  reach the surface is severity 1.
Medial OA, Lateral OA: cartilage loss of the femoral condyle or tibial plateau on that side.
PF OA: cartilage loss of the patella or trochlea.
Effusion: joint fluid. Synovitis: synovial thickening or inflammation.
Baker's: popliteal cyst. Contusion: bone bruise or marrow edema without a fracture line.
Fracture: fracture line or cortical break. An old or healed fracture is severity 1.

If the report covers both knees, use the knee the report is mainly about.
Return one JSON object with exactly these keys: """ + ", ".join(f'"{k}"' for k in LABELS) + """
Example value: "ACL": [4, 2, "complete tear of the ACL"]"""

def prompt(tok, report):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Report:\n" + report}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)

def as_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None

def parse(text):
    t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        d = json.loads(t)
    except Exception:
        return None
    out = {}
    for k in LABELS:
        v = d.get(k)
        if isinstance(v, list) and len(v) >= 2:
            out[k] = (as_int(v[0]), as_int(v[1]), v[2] if len(v) > 2 else "")
        else:
            out[k] = (None, None, "")
    return out

def score(sev, cert, ev):
    if sev is None or (sev == 0 and not str(ev).strip()):
        return 0.5
    if sev == 0:
        return 0.0
    return sev + 0.25 * (cert if cert is not None else 1)

def dev_report(parsed):
    sc = pd.DataFrame([{k: score(*p[k]) if p else np.nan for k in LABELS} for p in parsed])
    aucs = {k: roc_auc_score(dev[k], sc[k]) for k in LABELS
            if dev[k].nunique() == 2 and sc[k].notna().all()}
    print(pd.Series(aucs).round(3).to_string())
    print("macro:", round(np.mean(list(aucs.values())), 3))
    silent = {k: int(((dev[k] == 1) & (sc[k] == 0.5)).sum()) for k in LABELS}
    print("gold positive but report silent:", {k: v for k, v in silent.items() if v})
    print("contradictions (gold=1 but explicit normal, or gold=0 but severity >= 3):")
    for k in LABELS:
        for i, p in enumerate(parsed):
            if p is None:
                continue
            s, g = score(*p[k]), int(dev.loc[i, k])
            if (g == 1 and s == 0) or (g == 0 and s >= 3):
                print(f"  {k:<17} study {i:>2} gold={g} sev={p[k][0]} cert={p[k][1]} {str(p[k][2])[:70]!r}")
    return sc

N_SPLITS = 0

@torch.no_grad()
def gen_batch(model, tok, texts, max_new_tokens):
    global N_SPLITS
    enc = out = None
    try:
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    except torch.OutOfMemoryError:
        if len(texts) == 1:
            raise
    enc = out = None
    with torch.cuda.device(model.device):
        torch.cuda.empty_cache()
    N_SPLITS += 1
    h = len(texts) // 2
    return gen_batch(model, tok, texts[:h], max_new_tokens) + gen_batch(model, tok, texts[h:], max_new_tokens)

def run_par(reports, bs=16, max_new_tokens=400):
    order = np.argsort([len(r) for r in reports])
    batches = [order[i:i + bs] for i in range(0, len(order), bs)]
    outs = [None] * len(reports)
    def worker(w):
        m, t = models[w], toks[w]
        for idx in batches[w::len(models)]:
            texts = [prompt(t, reports[j]) for j in idx]
            for j, o in zip(idx, gen_batch(m, t, texts, max_new_tokens)):
                outs[j] = o
    with ThreadPoolExecutor(len(models)) as ex:
        list(ex.map(worker, range(len(models))))
    return outs

# Cell 4: Qwen3-4B on dev

In [ ]:
from sklearn.metrics import roc_auc_score
dev = gold[gold["split"] == "dev"].reset_index(drop=True)
t0 = time.time()
outs_dev = run_par(dev["Report"].tolist(), bs=10)
print(f"=== 4B, v3 prompt + D1 ({time.time() - t0:.0f}s) ===")
parsed_dev = [parse(t) for t in outs_dev]
print("parse failures:", sum(p is None for p in parsed_dev))
sc_dev = dev_report(parsed_dev)

# Cell 5: Qwen3-4B throughput

In [ ]:
silver = train[train[LABELS].isna().all(axis=1)].sample(64, random_state=0)
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)
N_SPLITS = 0
t0 = time.time()
outs_s = run_par(silver["Report"].tolist(), bs=16)
el = time.time() - t0
print(f"{el:.0f}s for 64 -> about {el / 64 * 4349 / 3600:.1f} h for all silver, OOM splits: {N_SPLITS}")
print("parse failures:", sum(parse(t) is None for t in outs_s))
print({i: f"peak {torch.cuda.max_memory_allocated(i) / 1e9:.1f} GB" for i in range(torch.cuda.device_count())})

# Cell 6: load Qwen3-8B in 4-bit (NF4), one copy per GPU

In [ ]:
!pip install -q bitsandbytes
import gc
from transformers import BitsAndBytesConfig

del models
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()

DIR_8B = [os.path.dirname(c) for c in cfgs if "/8b/" in c][0]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

toks, models = [], []
for i in range(torch.cuda.device_count()):
    t = AutoTokenizer.from_pretrained(DIR_8B)
    t.padding_side = "left"
    toks.append(t)
    models.append(AutoModelForCausalLM.from_pretrained(DIR_8B, quantization_config=bnb,
                                                       device_map={"": i}).eval())
print(DIR_8B)
print({i: f"{torch.cuda.memory_allocated(i) / 1e9:.1f} GB" for i in range(torch.cuda.device_count())})

# Cell 7: Qwen3-8B NF4 on dev (same code as cell 4)

In [ ]:
from sklearn.metrics import roc_auc_score
dev = gold[gold["split"] == "dev"].reset_index(drop=True)
t0 = time.time()
outs_dev = run_par(dev["Report"].tolist(), bs=10)
print(f"=== 8B nf4, v3 prompt + D1 ({time.time() - t0:.0f}s) ===")
parsed_dev = [parse(t) for t in outs_dev]
print("parse failures:", sum(p is None for p in parsed_dev))
sc_dev = dev_report(parsed_dev)

# Cell 8: Qwen3-8B NF4 throughput (same code as cell 5)

In [ ]:
silver = train[train[LABELS].isna().all(axis=1)].sample(64, random_state=0)
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)
N_SPLITS = 0
t0 = time.time()
outs_s = run_par(silver["Report"].tolist(), bs=16)
el = time.time() - t0
print(f"{el:.0f}s for 64 -> about {el / 64 * 4349 / 3600:.1f} h for all silver, OOM splits: {N_SPLITS}")
print("parse failures:", sum(parse(t) is None for t in outs_s))
print({i: f"peak {torch.cuda.max_memory_allocated(i) / 1e9:.1f} GB" for i in range(torch.cuda.device_count())})

# Cell 9: D3, evidence must be found verbatim

In [ ]:
import re

def norm(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

def verify(p, report):
    if p is None:
        return None
    r = norm(report)
    out = {}
    for k, (sev, cert, ev) in p.items():
        if sev == 0 and not (norm(ev) and norm(ev) in r):
            sev = None
        out[k] = (sev, cert, ev)
    return out

quotes = [(p[k][2], r) for p, r in zip(parsed_dev, dev["Report"]) if p for k in LABELS if norm(p[k][2])]
print(f"evidence found verbatim in report: {np.mean([norm(e) in norm(r) for e, r in quotes]):.1%} of {len(quotes)}")

print("=== 8B nf4, v3 prompt + D1 + D3 ===")
parsed_dev_v = [verify(p, r) for p, r in zip(parsed_dev, dev["Report"])]
sc_dev = dev_report(parsed_dev_v)

# Cell 10: why did throughput outputs fail to parse

In [ ]:
def parse_lenient(text):
    t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    out, hits = {}, 0
    for k in LABELS:
        m = re.search('"' + re.escape(k) + r'"\s*:\s*\[\s*(null|-?\d+)\s*,\s*(null|-?\d+)'
                      r'(?:\s*,\s*"((?:[^"\\]|\\.)*)")?', t)
        if m:
            hits += 1
            out[k] = (as_int(m.group(1)), as_int(m.group(2)), m.group(3) or "")
        else:
            out[k] = (None, None, "")
    return out, hits

for t in [t for t in outs_s if parse(t) is None]:
    n_tok = len(toks[0](t)["input_ids"])
    print(f"{n_tok} tokens, keys recovered by regex: {parse_lenient(t)[1]}/12")
    print("  ends with:", repr(t[-150:]))